# Validate prepared PhysioFSQ datasets

This notebook checks the released downstream array layout and metadata. It does not download data, rebuild windows, write over datasets, train models, or read entire arrays into memory.

Set `DOWNSTREAM_DATA_ROOT` to the prepared common data root and `DALIA_RR_DATA_ROOT` to the prepared cycle-derived DaLiA RR parent. Each root contains its task folders and root `manifest.json`.

In [ ]:
from pathlib import Path
import json, os
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
COMMON_ROOT = Path(os.environ.get("DOWNSTREAM_DATA_ROOT", PROJECT_ROOT / "data" / "downstream_125hz_v1"))
DALIA_RR_ROOT = Path(os.environ.get("DALIA_RR_DATA_ROOT", PROJECT_ROOT / "data" / "dalia_rr10s_cycle_v1"))
TASK_ROOTS = {
    "ptb-xl": COMMON_ROOT, "csn": COMMON_ROOT, "physionet2017": COMMON_ROOT,
    "ppg_dalia_hr": COMMON_ROOT, "bidmc_hr": COMMON_ROOT, "bidmc_rr": COMMON_ROOT,
    "uci_sbp": COMMON_ROOT, "uci_dbp": COMMON_ROOT,
    "ppg_dalia_rr": DALIA_RR_ROOT,
}
for root in (COMMON_ROOT, DALIA_RR_ROOT):
    root_manifest = root / "manifest.json"
    if not root_manifest.is_file():
        raise FileNotFoundError(f"Missing prepared-data root manifest: {root_manifest}")
    if json.loads(root_manifest.read_text(encoding="utf-8")).get("complete") is not True:
        raise ValueError(f"Prepared-data root is not marked complete: {root_manifest}")
SPLITS = ("train", "val", "test")


## Check manifests and array headers

NumPy uses memory-mapped headers here, so waveform and label arrays are not copied into RAM. The checks compare stored dimensions, offsets, metadata row counts, and task manifests. No waveform or label values are exported.

In [ ]:
def check_task(task, parent):
    task_dir = parent / task
    manifest_path = task_dir / "manifest.json"
    if not manifest_path.is_file():
        raise FileNotFoundError(f"Missing prepared manifest: {manifest_path}")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if manifest.get("complete") is not True:
        raise ValueError(f"{task}: manifest is not marked complete")
    rows = []
    for split in SPLITS:
        x_path = task_dir / f"{split}_features.npy"
        y_path = task_dir / f"{split}_labels.npy"
        offsets_path = task_dir / f"{split}_window_offsets.npy"
        lengths_path = task_dir / f"{split}_valid_lengths.npy"
        samples_path = task_dir / f"{split}_samples.csv"
        windows_path = task_dir / f"{split}_windows.csv"
        for path in (x_path, y_path, offsets_path, lengths_path, samples_path, windows_path):
            if not path.is_file():
                raise FileNotFoundError(f"{task}/{split}: missing {path.name}")
        x = np.load(x_path, mmap_mode="r", allow_pickle=False)
        y = np.load(y_path, mmap_mode="r", allow_pickle=False)
        offsets = np.load(offsets_path, mmap_mode="r", allow_pickle=False)
        lengths = np.load(lengths_path, mmap_mode="r", allow_pickle=False)
        with samples_path.open(encoding="utf-8") as stream:
            sample_rows = sum(1 for _ in stream) - 1
        with windows_path.open(encoding="utf-8") as stream:
            window_rows = sum(1 for _ in stream) - 1
        expected_channels = 12 if task == "ptb-xl" else 1
        if x.ndim != 3 or x.shape[1:] != (expected_channels, 1250):
            raise ValueError(f"{task}/{split}: unexpected feature shape {x.shape}")
        if len(offsets) != len(y) + 1 or int(offsets[0]) != 0 or int(offsets[-1]) != len(x):
            raise ValueError(f"{task}/{split}: window offsets do not align")
        if len(lengths) != len(x) or sample_rows != len(y) or window_rows != len(x):
            raise ValueError(f"{task}/{split}: array and metadata row counts disagree")
        rows.append({"task": task, "split": split, "samples": int(len(y)), "windows": int(len(x)),
                     "features_shape": tuple(x.shape), "labels_shape": tuple(y.shape),
                     "feature_dtype": str(x.dtype), "label_dtype": str(y.dtype), "status": "OK"})
    return rows

report = []
for task, parent in TASK_ROOTS.items():
    report.extend(check_task(task, parent))
validation = pd.DataFrame(report)
validation


## UCI preprocessing scope

The packaged UCI adapter only reuses a previously complete and verified 10-second product; it does not rebuild from raw MAT files. Set `UCI_VERIFIED_10S_ROOT` to that product and `UCI_RAW_SOURCE_ROOT` to the directory containing `data/Part_1.mat` through `data/Part_4.mat` for raw-byte SHA-256 checks. The UCI model runs consume the prepared 10-second arrays.